In [1]:
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
import importlib
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
warnings.filterwarnings('ignore')

In [2]:
trainset = h5py.File("../drive/MyDrive/vscode-ssh/GIZ_Biomass/data/09072022_1154_train.h5", "r")
validateset = h5py.File("../drive/MyDrive/vscode-ssh/GIZ_Biomass/data/09072022_1154_val.h5", "r")
testset = h5py.File("../drive/MyDrive/vscode-ssh/GIZ_Biomass/data/09072022_1154_test.h5", "r")

In [3]:
# attributes of trainset
trainset.keys()

<KeysViewHDF5 ['agbd', 'cloud', 'images', 'lat', 'lon', 'scl']>

# Prediction using CNN

In [4]:
# train
train_images = np.array(trainset['images'],dtype=np.float64)
train_biomasses = np.array(trainset['agbd'],dtype=np.float64).reshape(-1, 1)

# validate
validate_images = np.array(validateset['images'],dtype=np.float64)
validate_biomasses = np.array(validateset['agbd'],dtype=np.float64).reshape(-1, 1)

# test 
test_images = np.array(testset['images'],dtype=np.float32)
test_biomasses = np.array(testset['agbd'],dtype=np.float32).reshape(-1, 1)

In [5]:
class CustomScaler(BaseEstimator, TransformerMixin):
  
  def fit(self, X, y=None):
    self.mean = np.mean(X, axis=(0,1,2))
    self.std = np.std(X, axis=(0,1,2))
    return self

  def transform(self, X, y=None):
    return (X-self.mean[None,None,None,:])/self.std[None,None,None,:] 

  def reverse_transform(self):
    return None

scaler = CustomScaler()
train_images = scaler.fit_transform(train_images)
validate_images = scaler.transform(validate_images)
test_images = scaler.transform(test_images)

scaler = StandardScaler()
train_biomasses = scaler.fit_transform(train_biomasses)
validate_biomasses = scaler.transform(validate_biomasses)
test_biomasses = scaler.transform(test_biomasses)

In [10]:
model = models.Sequential([
    layers.Input(shape=(15, 15, 12)),

    layers.Conv2D(filters=32, kernel_size = [3, 3], activation='relu'),
    layers.Conv2D(filters=64, kernel_size = [3, 3], activation='relu'),
    layers.Conv2D(filters=128, kernel_size = [3, 3], activation='relu'),
    layers.Conv2D(filters=128, kernel_size = [3, 3], activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),

    layers.Flatten(),

    layers.Dense(512, activation="relu"),
    layers.Dense(256, activation="relu"),

    layers.Dense(1, activation="linear")
])

model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_14 (Conv2D)          (None, 13, 13, 32)        3488      
                                                                 
 conv2d_15 (Conv2D)          (None, 11, 11, 64)        18496     
                                                                 
 conv2d_16 (Conv2D)          (None, 9, 9, 128)         73856     
                                                                 
 conv2d_17 (Conv2D)          (None, 7, 7, 128)         147584    
                                                                 
 max_pooling2d_9 (MaxPooling  (None, 3, 3, 128)        0         
 2D)                                                             
                                                                 
 flatten_3 (Flatten)         (None, 1152)              0         
                                                      

In [11]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

history = model.fit(
    x = train_images,
    y = train_biomasses,
    validation_data=(validate_images, validate_biomasses),
    epochs=100,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        ),
        tf.keras.callbacks.TensorBoard(
            './logs', update_freq=1
        )
    ]
)

Epoch 1/100
783/783 [==============================] - ETA: 0s - loss: 0.9369 - root_mean_squared_error: 0.9679